In [1]:
from helper_functions import LSE, construct_R, decomp_orthog
import numpy as np
import gurobipy as gp
from gurobipy import GRB
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt  

In [2]:
#This function, given an NxI np-array Mat, creates a new NXM np array Mat_twoway which contains all main-effects, quadratic terms, and
#two-way interactions of the columns in Mat
def two_way_interaction(Mat):

    #Retrieve the number of columns in Mat
    num_cols_Mat = np.shape(Mat)[1]
    
    Mat_twoway = np.concatenate((Mat,Mat**2),axis = 1)
    for i in range(num_cols_Mat):
        for j in range(i+1, num_cols_Mat):
            interact_ij = np.array([Mat[:,i]*Mat[:,j]]).T
            Mat_twoway = np.concatenate((Mat_twoway,interact_ij), axis = 1)

    return Mat_twoway

In [3]:
#Constructs a list to be used in decomp_orthog to indicate heredity relations. We assume a full quadratics-interactions effects model is being fit. We assume that
#there is no constant term in the model.
def heredity_list(I,J):
    #I - number of input variables
    #J - number of objective function
    
    inner_list = []

    #Main effects
    for i in range(I):
        inner_list.append([])

    #Quadratics
    for i in range(I):
        inner_list.append([i])

    #Interactions
    for i in range(I):
        for i_2 in range(i+1,I):
            inner_list.append([i,i_2])

    D_her = [inner_list for j in range(J)]

    return D_her

In [4]:
#This function takes the scaling factors from decomp_orthog and multiplies them by the LSE estimates to get the scaled coefficients.
def decomp_scaled_coefficients(theta,lse_coeff):
    #Theta: this should be an MxJ numpy array of scaling factors obtained from decomp_orthog.
    #lse_coeff: this should be an MxJ numpy array of estimated coefficients coming from LSE, and should be the LSE estimates used in constructing matrices R_1,...
    #R_J for decomp_orthog.

    return theta*lse_coeff

In [28]:
#Import the csv file containing inputs and responses of tradespace simulator
tradespace_sim_orig = pd.read_csv("tradespace_simulated_data.csv")
tradespace_sim = (tradespace_sim_orig-tradespace_sim_orig.mean())/tradespace_sim_orig.std()
print(tradespace_sim)

     frame_axle_mount_inset  frame_underbody_length  frame_underbody_width  \
0                  0.481368               -1.152814              -0.839295   
1                 -0.212579               -0.456733               1.293394   
2                 -0.443895                1.666312              -1.085375   
3                 -1.137843               -0.491537               0.309076   
4                  0.250052                1.248664              -1.577534   
..                      ...                     ...                    ...   
995                1.406632                1.666312              -0.757269   
996                0.712684                1.039840              -0.839295   
997                1.175316               -1.535658               0.637182   
998                0.712684               -0.665557               1.047315   
999                1.175316               -0.491537               0.965288   

     offload_generator_height  offload_generator_length  \
0   

In [6]:
#Create the model matrix. We will use a full response surface model for each response. Recall that running_gear_track_thickness and running_gear_track_width
#both only have two levels, so we CAN NOT ESTIMATE QUADRATIC EFFECTS FOR THESE two predictors.

design_matrix = tradespace_sim.iloc[:,0:13].to_numpy()

model_matrix = two_way_interaction(design_matrix)
#print(model_matrix[:,22])
model_matrix = np.delete(model_matrix,[22],axis = 1) #Get rid of the quadratic effects for running_gear_track_width

model_matrix_with_intercept = np.concatenate((np.ones((1000,1)),model_matrix),axis = 1)
#print(model_matrix_with_intercept)

In [7]:
#Fit linear models for each of the 4 responses. First include the intercept, then subtract it from the responses and refit.
#print(tradespace_sim.iloc[:,14].to_numpy().reshape(1000,1))
R_back_deck_int,LSE_back_deck_int = construct_R(model_matrix_with_intercept,tradespace_sim.iloc[:,13].to_numpy().reshape(1000,))
R_running_gear_int,LSE_running_gear_int = construct_R(model_matrix_with_intercept,tradespace_sim.iloc[:,14].to_numpy().reshape(1000,))
R_smet_fcc_length_int,LSE_smet_fcc_length_int = construct_R(model_matrix_with_intercept,tradespace_sim.iloc[:,15].to_numpy().reshape(1000,))
R_smet_fcc_turn_int,LSE_smet_fcc_turn_int = construct_R(model_matrix_with_intercept,tradespace_sim.iloc[:,16].to_numpy().reshape(1000,))

#print(LSE_back_deck_int)
#print(LSE_running_gear_int)
#print(LSE_smet_fcc_length_int)
#print(LSE_smet_fcc_turn_int)

In [8]:
#Subtract off intercept values
tradespace_sim['back_deck_overhang_in_center'] = tradespace_sim['back_deck_overhang_in'] - LSE_back_deck_int[0]
tradespace_sim['running_gear_contact_patch_area_in2_center'] = tradespace_sim['running_gear_contact_patch_area_in2'] - LSE_running_gear_int[0]
tradespace_sim['smet_fcc_length_in_center'] = tradespace_sim['smet_fcc_length_in'] - LSE_smet_fcc_length_int[0]
tradespace_sim['smet_fcc_ctoc_turning_diameter_in_center'] = tradespace_sim['smet_fcc_ctoc_turning_diameter_in'] - LSE_smet_fcc_turn_int[0]
#print(tradespace_sim)

In [9]:
#Fit models without intercept term
R_back_deck_noint,LSE_back_deck_noint = construct_R(model_matrix,tradespace_sim.iloc[:,17].to_numpy().reshape(1000,))
R_running_gear_noint,LSE_running_gear_noint = construct_R(model_matrix,tradespace_sim.iloc[:,18].to_numpy().reshape(1000,))
R_smet_fcc_length_noint,LSE_smet_fcc_length_noint = construct_R(model_matrix,tradespace_sim.iloc[:,19].to_numpy().reshape(1000,))
R_smet_fcc_turn_noint,LSE_smet_fcc_turn_noint = construct_R(model_matrix,tradespace_sim.iloc[:,20].to_numpy().reshape(1000,))

#print(LSE_back_deck_noint)
#print(LSE_running_gear_noint)
#print(LSE_smet_fcc_length_noint)
#print(LSE_smet_fcc_turn_noint)

#print(R_back_deck_noint)
#print(R_running_gear_noint)
#print(R_smet_fcc_length_noint)
#print(R_smet_fcc_turn_noint)

In [10]:
#Need to make heredity constraints. Make sure to remove the term [9] from each list, as this corresponds to heredity constraint for
#I(running_gear_track_width**2), which is not in our model (because it only has two continuous levels)
full_her = heredity_list(13,4)
actual_her = [[] for i in range(4)]
len_her = len(full_her[0])
for i in range(4):
    for j in range(len_her):
        if j != 22:
            actual_her[i].append(full_her[i][j])

#print(actual_her)

In [11]:
#
clusters = 2
responses = tradespace_sim.iloc[:,17:21].to_numpy().T
#print(responses)
#print(tradespace_sim)
decomp_fit = decomp_orthog(responses,[R_back_deck_noint,R_running_gear_noint,R_smet_fcc_length_noint,R_smet_fcc_turn_noint],actual_her,clusters,t=300,focus=0,outputflag = 1)

Set parameter Username

--------------------------------------------
--------------------------------------------

Academic license - for non-commercial use only - expires 2025-05-09
Set parameter TimeLimit to value 300
Gurobi Optimizer version 11.0.3 build v11.0.3rc0 (win64 - Windows 10.0 (19045.2))

CPU model: 11th Gen Intel(R) Core(TM) i9-11900H @ 2.50GHz, instruction set [SSE2|AVX|AVX2|AVX512]
Thread count: 8 physical cores, 16 logical processors, using up to 16 threads

Optimize a model with 678 rows, 432 columns and 1360 nonzeros
Model fingerprint: 0xdb49c1ed
Model has 21424 quadratic objective terms
Model has 18 quadratic constraints
Variable types: 418 continuous, 14 integer (14 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  QMatrix range    [1e+00, 1e+00]
  QLMatrix range   [1e+00, 1e+00]
  Objective range  [8e-18, 2e+03]
  QObjective range [2e-09, 2e+03]
  Bounds range     [1e+00, 4e+02]
  RHS range        [1e+00, 1e+00]
  QRHS range       [1e+00, 1e+00]


In [12]:
#save coefficients and print
decomp_coeff = decomp_scaled_coefficients(decomp_fit[0].T,np.array([LSE_back_deck_noint,LSE_running_gear_noint,LSE_smet_fcc_length_noint,LSE_smet_fcc_turn_noint]).T)
print(decomp_coeff)
print(decomp_fit[0])
print(decomp_fit[1])

[[-6.78364331e-01 -0.00000000e+00 -0.00000000e+00 -0.00000000e+00]
 [ 0.00000000e+00  9.46658260e-01  9.70871958e-01  7.96789812e-01]
 [ 0.00000000e+00 -5.63586019e-17  4.88649161e-03  5.47835775e-01]
 [ 0.00000000e+00  1.26740451e-16 -5.86784942e-03 -0.00000000e+00]
 [ 0.00000000e+00 -1.68921379e-16 -1.23561362e-03 -0.00000000e+00]
 [-0.00000000e+00 -5.98722809e-17 -5.47096005e-03 -0.00000000e+00]
 [ 7.37479997e-01 -0.00000000e+00  0.00000000e+00  0.00000000e+00]
 [-0.00000000e+00  0.00000000e+00  8.54472460e-02 -0.00000000e+00]
 [ 0.00000000e+00  5.02545242e-02  1.01192376e-02  0.00000000e+00]
 [ 0.00000000e+00  8.57563759e-02  1.11299299e-02  4.40130783e-02]
 [ 0.00000000e+00 -6.64990422e-02  9.47143246e-03  0.00000000e+00]
 [ 0.00000000e+00  1.40751787e-17  1.17376263e-02  0.00000000e+00]
 [ 0.00000000e+00 -4.16333634e-16  3.97130053e-02  0.00000000e+00]
 [ 1.05056461e-01 -0.00000000e+00  0.00000000e+00 -0.00000000e+00]
 [ 0.00000000e+00 -4.42509578e-17  1.57575989e-03  0.00000000e

In [13]:
#We need to calculate the adjusted R^2 for each model after clustering. This adjusted R^2 is for the case where there is NO INTERCEPT.
def adjusted_r2(mod_matrix,coeff,response):

    n = len(response)
    p = len(coeff)
    rss = np.sum(np.square(mod_matrix@coeff - response))
    tss = np.sum(np.square(response))

    r2 = 1 - rss/tss

    adj_r2 = 1 - ((1-r2)*(n)/(n-p))

    return adj_r2

In [14]:
#back_deck
print('Decomp Back Deck R2: '+ str(adjusted_r2(model_matrix,decomp_coeff[:,0],tradespace_sim.iloc[:,17].to_numpy().reshape(1000,))))
print('Original Back Deck R2: ' + str(adjusted_r2(model_matrix,LSE_back_deck_noint,tradespace_sim.iloc[:,17].to_numpy().reshape(1000,))))

#running_gear
print('Decomp Running Gear R2:'+ str(adjusted_r2(model_matrix,decomp_coeff[:,1],tradespace_sim.iloc[:,18].to_numpy().reshape(1000,))))
print('Original Running Gear R2:'+str(adjusted_r2(model_matrix,LSE_running_gear_noint,tradespace_sim.iloc[:,18].to_numpy().reshape(1000,))))

#smet_fcc_length
print('Decomp Smet fcc length R2:'+ str(adjusted_r2(model_matrix,decomp_coeff[:,2],tradespace_sim.iloc[:,19].to_numpy().reshape(1000,))))
print('Original Smet fcc length R2:'+str(adjusted_r2(model_matrix,LSE_smet_fcc_length_noint,tradespace_sim.iloc[:,19].to_numpy().reshape(1000,))))

#smet_fcc_turn
print('Decomp Smet fcc turn R2:' + str(adjusted_r2(model_matrix,decomp_coeff[:,3],tradespace_sim.iloc[:,20].to_numpy().reshape(1000,))))
print('Original Smet fcc turn R2:' + str(adjusted_r2(model_matrix,LSE_smet_fcc_turn_noint,tradespace_sim.iloc[:,20].to_numpy().reshape(1000,))))

Decomp Back Deck R2: 0.9809375346999258
Original Back Deck R2: 0.9828875382169227
Decomp Running Gear R2:0.9068113818316625
Original Running Gear R2:1.0
Decomp Smet fcc length R2:0.9443732068341666
Original Smet fcc length R2:0.9990798667226266
Decomp Smet fcc turn R2:0.934096768910525
Original Smet fcc turn R2:1.0


In [51]:
def tradespace_optimization(linear_coef,quad_coef,interact_coef,mean,sd):
    #linear_coef,quad_coef,interact_coef: vector of coefficients for the objective terms. The objective will have the form
    #c_1 x_1 + c_2 x_2 + ... c_13 x_13 + d_1 x_1**2 + d_2 x_2**2 + ... + d_9 x_9**2 + d_11 x_11**2 + d_12 x_12**2 + d_13 x_13**2 + (NO d_10 * x_10**2 !!!)
    #h_1_2 x_1*x_2 +....+ h_1_13 x_1*x_13 + h_2_3 x_2*x_3 + ... + h_12_13 x_12*x_13
    #linear coef is a list, quad_coef is a list, and interact_coef is a np matrix. This procedure is used to solve the weighted sum problem.
    #Weights of the weighted sum problem should be incorporated into the coefficients. Make sure to negate coefficients for running gear contact area!

    #mean: mean values of each of the predictor variables
    #sd: standard deviations of each of the predictor variables (recall we standardized the predictor variables and responses).
    model = gp.Model("tradespace_optimization")
    
    x = model.addVars(13, vtype=GRB.CONTINUOUS, name="x")

    #Constraints for each of the variables. Recall that we standardized the variables for our regression model.
    #frame_axle_mount_inset
    x[0].lb = (1 - mean[0])/sd[0]
    x[0].ub = (15 - mean[0])/sd[0]

    #frame_underbody_length
    x[1].lb = (34 - mean[1])/sd[1]
    x[1].ub = (132 - mean[1])/sd[1]

    #frame_underbody_width
    x[2].lb = (16 - mean[2])/sd[2]
    x[2].ub = (58 - mean[2])/sd[2]

    #offload_generator_height
    x[3].lb = (10 - mean[3])/sd[3]
    x[3].ub = (40 - mean[3])/sd[3]

    #offload_generator_length
    x[4].lb = (10 - mean[4])/sd[4]
    x[4].ub = (40 - mean[4])/sd[4]

    #offload_generator_width
    x[5].lb = (10 - mean[5])/sd[5]
    x[5].ub = (40 - mean[5])/sd[5]

    #running_gear_back_mount_inset
    x[6].lb = (5 - mean[6])/sd[6]
    x[6].ub = (20 - mean[6])/sd[6]

    #running_gear_front_mount_inset
    x[7].lb = (5 - mean[7])/sd[7]
    x[7].ub = (20 - mean[7])/sd[7]

    #running_gear_drive_gear_radius
    x[8].lb = (1 - mean[8])/sd[8]
    x[8].ub = (3 - mean[8])/sd[8]

    #running_gear_track_width
    x[9].lb = (14 - mean[9])/sd[9]
    x[9].ub = (15 - mean[9])/sd[9]

    #running_gear_road_wheel_radius
    x[10].lb = (1 - mean[10])/sd[10]
    x[10].ub = (5 - mean[10])/sd[10]

    #running_gear_wheel_radius
    x[11].lb = (11 - mean[11])/sd[11]
    x[11].ub = (20 - mean[11])/sd[11]

    #winch_stowed_width
    x[12].lb = (2 - mean[12])/sd[12]
    x[12].ub = (9 - mean[12])/sd[12]

    model.setObjective(sum([linear_coef[i]*x[i] for i in range(13)]) + quad_coef[0]*x[0]*x[0] + quad_coef[1]*x[1]*x[1] + quad_coef[2]*x[2]*x[2] +
    quad_coef[3]*x[3]*x[3] + quad_coef[4]*x[4]*x[4] + quad_coef[5]*x[5]*x[5] + quad_coef[6]*x[6]*x[6] + quad_coef[7]*x[7]*x[7] + quad_coef[8]*x[8]*x[8] +
    quad_coef[9]*x[10]*x[10] + quad_coef[10]*x[11]*x[11] + quad_coef[11]*x[12]*x[12] + 
    sum([interact_coef[i,j-1]*x[i]*x[j] for i in range(12) for j in range(i+1,13)],GRB.MINIMIZE))
    
    
    model.optimize()

    x_list = [x[i].X for i in range(13)]

    return [x_list,model.objVal]

In [52]:
#Optimize back_deck_overhang

#Create variables
bd_linear = decomp_coeff[0:13,0]
bd_quad = decomp_coeff[13:25,0]
print(bd_linear)
print(bd_quad)
bd_int = np.zeros((12,12))
list_blah = [i for i in range(1,100)]
count_var = 0
for i in range(12):
    for j in range(i+1,13):
        bd_int[i,j-1] = decomp_coeff[25+count_var,0]
        count_var = count_var + 1
print(bd_int)

mean_pred = tradespace_sim_orig.mean().to_numpy()
sd_pred = tradespace_sim_orig.std().to_numpy()

print(tradespace_optimization(bd_linear,bd_quad,bd_int,mean_pred,sd_pred))

[-0.67836433  0.          0.          0.          0.         -0.
  0.73748    -0.          0.          0.          0.          0.
  0.        ]
[ 0.10505646  0.          0.          0.         -0.          0.
  0.12265651 -0.         -0.          0.          0.          0.        ]
[[ 0.          0.          0.         -0.          0.         -0.22212388
   0.         -0.          0.         -0.         -0.         -0.        ]
 [ 0.          0.         -0.         -0.         -0.         -0.
  -0.         -0.          0.          0.          0.          0.        ]
 [ 0.          0.          0.          0.          0.         -0.
   0.          0.         -0.          0.         -0.          0.        ]
 [ 0.          0.          0.         -0.          0.          0.
   0.          0.          0.          0.         -0.          0.        ]
 [ 0.          0.          0.          0.         -0.          0.
   0.          0.         -0.         -0.          0.         -0.        ]
 [ 0